# Monte Carlo propagation

The measurements (ultimate analysis, thermogram and gas composition) are resampled within their uncertainty and the program is re-solved from the reference state (Section 3.6). The article uses 300 draws per feedstock; 40 draws are used here to keep the run short.

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path
ARTICLE = Path.cwd().parent / "results"
OUTPUT = Path.cwd() / "output" / "04_monte_carlo"
OUTPUT.mkdir(parents=True, exist_ok=True)
if (ARTICLE / "best_vectors.npz").exists():
    handle, temporary = tempfile.mkstemp(dir=OUTPUT, suffix=".npz")
    os.close(handle)
    shutil.copyfile(ARTICLE / "best_vectors.npz", temporary)
    os.replace(temporary, OUTPUT / "best_vectors.npz")
os.environ.setdefault("CCPYRO_RESULTS", str(OUTPUT))
os.environ.setdefault("CCPYRO_FIGDATA", str(OUTPUT / "figdata"))
try:
    import ccpyro  # installed with `pip install -e .`
except ImportError:  # running from a clone without installation
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ccpyro
import numpy as np
import pandas as pd
np.set_printoptions(precision=6, suppress=True)
print("ccpyro", ccpyro.__version__)

In [ ]:
from ccpyro import model as mp
built = mp.build_cases(random_starts=0)
label = "rice husk (pelletized)"
ctx, st = built["contexts"][label], built["states"][label]
rng = np.random.default_rng(mp.RANDOM_SEED)
draws = mp.propagate(ctx, st, built["relative_sd"], 40, rng)
summary = mp.summarise(draws, mp.collect(st, ctx))
summary[["T", "w_tar", "w_tar_sd", "x_t", "x_t_sd", "y_t", "y_t_sd", "CGE", "CGE_sd"]].round(4)

In [ ]:
published = ARTICLE / "product_yields.csv"
if published.exists():
    ref = pd.read_csv(published)
    ref = ref[ref.treatment == label][["T", "w_tar_sd", "x_t_sd", "y_t_sd", "CGE_sd"]]
    print("standard deviations with 300 draws (article):")
    print(ref.round(4).to_string(index=False))